In [ ]:
import dataclasses

import flax.nnx as nnx
import jax
import jax.numpy as jnp
from jax import lax

from jax.typing import ArrayLike
from typing import NamedTuple

In [ ]:
"""
Gated DeltaNet-2 — chunkwise parallel training core (JAX), ANNOTATED.

Every line of the algorithm is mapped to the equations of
Hatamizadeh, Choi, Kautz, "Gated DeltaNet-2: Decoupling Erase and Write in
Linear Attention" (arXiv:2605.22791). Dual numbering is given as
"main text / Appendix A" where the equation appears in both.

State orientation follows the paper: S in R^{dk x dv}, output o_t = S_t^T q_t.

Per-head recurrence (Eq. 10 / 29):
    S_r = (I - k_r e_r^T) diag(alpha_r) S_{r-1} + k_r z_r^T,
    e_r = b_r ⊙ k_r,   z_r = w_r ⊙ v_r,   alpha_r = exp(g_r).

Chunkwise WY form (Eqs. 18-25 / 30-44):
    G_r   = cumsum(g)             (inclusive, within chunk)        Eq. 18/30
    gamma = exp(G),  gamma_C = gamma[-1]                           Eq. 18/30
    Kbar  = gamma^{-1} ⊙ K        (decay-normalized keys)          Eq. 19/32/33
    Ebar  = gamma     ⊙ (B ⊙ K)   (decay-absorbed erase factor)    Eq. 20/33
    Z     = W ⊙ V                                                  Eq. 20/33
    T     = tril(Ebar Kbar^T, -1)                                  Eq. 21/34
    A     = (I + T)^{-1}          (unit lower-triangular solve)    Eq. 21/34
    Y, U  = A Ebar, A Z           (WY auxiliaries; share inverse)  Eq. 22/34
    R     = U - Y S0                                               Eq. 35
    O     = Qgamma S0 + Aqk R                                      Eq. 24/44
    S_C   = diag(gamma_C) S0 + Ktail^T R                           Eq. 23/40

The gate-aware backward (Eqs. 64-82, Appendix B) is intentionally NOT written:
jax.grad differentiates straight through solve_triangular and the elementwise
gate products and reconstructs exactly those vector-Jacobian products. The
hand-derived backward is only needed for a fused Triton/Pallas kernel.
"""

# math runs in fp32 (paper App. D.1/D.3/D.4)
D_TYPE = jnp.float32


# --------------------------------------------------------------------------- #
#  Single (batch, head) sequence — the actual algorithm.
#  Everything else is vmap over (B, H) on top of this.
# --------------------------------------------------------------------------- #
def _chunkwise_single(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    g: jax.Array,
    b: jax.Array,
    w: jax.Array,
    S0: jax.Array,
    chunk_size: int,
) -> tuple[jax.Array, jax.Array]:
    """q,k,g,b: [L, dk]  v,w: [L, dv]  S0: [dk, dv]  ->  (O: [L, dv], S_final: [dk, dv])."""
    L, dk = k.shape
    dv = v.shape[-1]
    C = chunk_size
    N = L // C

    def to_chunks(x):
        return x.reshape(N, C, x.shape[-1]).astype(D_TYPE)

    q, k, v = to_chunks(q), to_chunks(k), to_chunks(v)
    g, b, w = to_chunks(g), to_chunks(b), to_chunks(w)

    eye = jnp.eye(C, dtype=D_TYPE)
    S0 = S0.astype(D_TYPE)

    def chunk_step(S, inp):
        # S is the raw chunk-entry state S_[n] (== S_0, NOT decay-normalized).
        # The per-chunk cumsum below resets every step, which realizes both
        # gamma_0 = 1 (Eq. 18/30) and the normalized init Ŝ_0 = S_[n] (Eq. 31 / A.1).
        qc, kc, vc, gc, bc, wc = inp  # each [C, d*]

        # --- Cumulative decay -------------------------------------------------
        # Eq. 18/30:  G_r = Σ_{i≤r} g_i (inclusive)
        G = jnp.cumsum(gc, axis=0)

        # Eq. 18/30:  γ_r = exp(G_r)
        gamma = jnp.exp(G)

        # γ_C, total chunk decay (last row); Eq. 40/41
        gamma_C = gamma[-1]

        # --- Decay normalization (removes Diag(α) from the recurrence) --------
        # Eq. 19/32/33:  K̄ = γ^{-1} ⊙ K  (exp(-G) = 1/γ in log-space)
        Kbar = kc * jnp.exp(-G)

        # Eq. 20/33:     Ē = γ ⊙ (B ⊙ K);  bc*kc = e_r (Eq. 8), γ⊙ = ē_r (Eq. 19/32)
        Ebar = gamma * (bc * kc)

        # Eq. 8, 20/33:  Z = W ⊙ V  (z_r = w_r ⊙ v_r)
        Z = wc * vc

        # Eq. 24/43:     Q_γ, row γ_r ⊙ q_r
        Qg = gamma * qc

        # --- WY triangular solve (the parallelization) ------------------------
        # Eq. 21/34, entry Eq. 87:  T = tril(Ē K̄ᵀ, -1), T_rs = ē_rᵀ k̄_s (s<r)
        T = jnp.tril(Ebar @ Kbar.T, k=-1)

        # Eq. 21/34:  A = (I + T)^{-1}  (unit lower-tri -> forward substitution)
        A = jax.scipy.linalg.solve_triangular(
            eye + T, eye, lower=True, unit_diagonal=True
        )

        # --- WY auxiliaries + residual ----------------------------------------
        # Eq. 22/34:  Y = A Ē  (erase-side auxiliary)
        Y = A @ Ebar

        # Eq. 22/34:  U = A Z  (write-side aux; SAME inverse A, two RHS — A.4)
        U = A @ Z

        # Eq. 35:     R = U − Y S_0  (stacked residual rows ρ_r, Eq. 37)
        R = U - Y @ S

        # --- Output block -----------------------------------------------------
        # Eq. 25/43:  (A_qk)_rs = 1_{r≥s} q_rᵀ Diag(γ_r/γ_s) k_s  (tril incl. diag: s≤r)
        Aqk = jnp.tril(Qg @ Kbar.T)

        # Eq. 24/44:  O = Q_γ S_0 + A_qk (U − Y S_0)
        o = Qg @ S + Aqk @ R

        # --- End-of-chunk state -----------------------------------------------
        # Eq. 23/41:  (K_tail)_r = (γ_C / γ_r) ⊙ k_r
        Ktail = kc * (gamma_C[None, :] / gamma)

        # Eq. 23/40:  S_[n+1] = Diag(γ_C) S_0 + K_tailᵀ R
        # Diag(γ_C) S_0: broadcast over key-channel rows (decay lives on key axis)
        S_new = gamma_C[:, None] * S + Ktail.T @ R

        return S_new, o

    # Cross-chunk recurrence: sequential scan over N = L/C chunks (Sec. 2.1 / Eq. 3 structure).
    S_final, o = lax.scan(chunk_step, S0, (q, k, v, g, b, w))
    return o.reshape(L, dv), S_final


def _recurrent_single(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    g: jax.Array,
    b: jax.Array,
    w: jax.Array,
    S0: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    """Token-by-token reference (Eq. 9 / 29). Same signature as the chunkwise core.

    Three-line factored form of Eq. 9, algebraically equal to the
    (I - k_t e_t^T) Diag(α_t) form of Eq. 10/29. O(L·dk·dv), no triangular solve —
    a trustworthy ground truth for verifying the chunkwise path.
    """
    q = q.astype(D_TYPE)
    k = k.astype(D_TYPE)
    v = v.astype(D_TYPE)
    g = g.astype(D_TYPE)
    b = b.astype(D_TYPE)
    w = w.astype(D_TYPE)
    S0 = S0.astype(D_TYPE)

    alpha = jnp.exp(g)  # Eq. 12/30:  α_r = exp(g_r)
    e = b * k  # Eq. 8:      e_r = b_r ⊙ k_r
    z = w * v  # Eq. 8:      z_r = w_r ⊙ v_r

    def step(S, inp):
        qt, kt, at, et, zt = inp

        qt = qt[:, None]
        kt = kt[:, None]
        at = at[:, None]
        et = et[:, None]
        zt = zt[:, None]

        # Eq. 9:  S̄_t = D_t S_{t-1} = Diag(α_t) S_{t-1} (scale key-channel rows)
        S_bar = at * S

        # Eq. 9:  r_t = S̄_tᵀ e_t  (read old content along erase direction)
        r_t = S_bar.T @ et

        # Eq. 9/15:  S_t = S̄_t + k_t (z_t − r_t)ᵀ  (rank-one delta write)
        S_new = S_bar + kt * (zt - r_t).T

        # Eq. 1:  o_t = S_tᵀ q_t
        o_t = S_new.T @ qt

        return S_new, o_t

    S_final, o = lax.scan(step, S0, (q, k, alpha, e, z))
    return o.squeeze(-1), S_final


# --------------------------------------------------------------------------- #
#  Batched public entry points: inputs are [B, H, L, d].  (No equations here —
#  pure plumbing: vmap the per-head algorithm over heads, then over batch.)
# --------------------------------------------------------------------------- #
def _batchify(fn: (...)) -> ...:
    # vmap over heads (axis 1) then batch (axis 0); S0 has no L axis.
    over_heads = jax.vmap(fn)
    return jax.vmap(over_heads)


def chunkwise_gated_delta_rule_2(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    g: jax.Array,
    b: jax.Array,
    w: jax.Array,
    S0: jax.Array,
    chunk_size: int = 64,
) -> tuple[jax.Array, jax.Array]:
    """Parallel chunkwise forward.

    q, k, g, b : [B, H, L, dk]      v, w : [B, H, L, dv]      S0 : [B, H, dk, dv]
    returns (O : [B, H, L, dv], S_final : [B, H, dk, dv]).
    """

    def fun(
        Q: jax.Array,
        K: jax.Array,
        V: jax.Array,
        G: jax.Array,
        B: jax.Array,
        W: jax.Array,
        So: jax.Array,
    ) -> tuple[jax.Array, jax.Array]:
        return _chunkwise_single(Q, K, V, G, B, W, So, chunk_size=chunk_size)

    return _batchify(fun)(q, k, v, g, b, w, S0)


def recurrent_gated_delta_rule_2(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    g: jax.Array,
    b: jax.Array,
    w: jax.Array,
    S0: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    """Token-by-token reference forward, same I/O as the chunkwise version."""
    return _batchify(_recurrent_single)(q, k, v, g, b, w, S0)


In [ ]:
"""
Gated DeltaNet-2 token-mixer layer in Flax NNX, ANNOTATED against the paper
(arXiv:2605.22791): Section 3.5 (block design) and Appendix C.1 (layer
parameterization), with supporting equations 11, 12, 85, 86 and the numerical
notes in Appendix D.

Block design (Fig. 1 right; Sec. 3.5 "Gated DeltaNet-2 token mixer"):
  q,k = L2norm(SiLU(ShortConv(Linear(x))))      # key-side paths + L2 norm (Sec. 3.5, App. D.2)
  v   =        SiLU(ShortConv(Linear(x)))        # value path (Sec. 3.5; Fig. 1 caption)
  g   = -exp(a) ⊙ softplus(Linear_f(x) + delta)  # log-decay, fp32 (Eq. 12 / 86, App. D.1)
  b   = sigmoid(Linear_b(x))                     # erase gate (Eq. 11 / 85); x2 if neg-eigenvalue
  w   = sigmoid(Linear_w(x))                     # write gate (Eq. 11 / 85)
  O   = chunkwise_gated_delta_rule_2(q,k,v,g,b,w, state)   # Gated Delta Rule-2 (Eq. 10)
  out = Linear_o( RMSNorm(O) * SiLU(gate) )      # gated RMSNorm + out proj (Sec. 3.5, App. D.5)

Grouped value heads (Sec. 3.5 last sentence / App. C.1): with num_v_heads = G*num_heads,
the key-side tensors q, k, the log-decay g, and b are repeated across the G value-head
groups; v and w already live on the value-head axis.

Scope: this is the recurrent TOKEN MIXER only (Fig. 1 right). The recurrent model
(Sec. 3.5 "Model families") stacks [this + MLP]; the hybrid model inserts
Sliding-Window Attention after it, repeating the cell [GDN-2, MLP, SWA, MLP]
(Fig. 1 left). Those wrappers are not implemented here.

Two honest deviations from the paper, flagged inline below:
  (1) A_log is stored per (head, key-channel); App. C.1 stores 'a' per key HEAD and
      broadcasts it over d_k. This implementation is a strict generalization (tie the
      d_k columns to recover the paper).
  (2) All Linear kernels use the paper's Xavier-uniform init, gain 2^{-2.5}, with zero
      biases (App. D.5). The one exception: the decay bias δ starts negative (−4), not
      the paper's value, to keep early decay mild for fp32 stability (App. D.1).
"""

F32 = jnp.float32

# App. D.5: Xavier-uniform init with gain 2^{-2.5} (variance_scaling scale = gain² =
# 2^{-5}), replacing Flax NNX's default Linear kernel init. Biases stay at zero (the
# NNX default) — except the decay bias δ, set negative in __init__ for fp32 safety.
_XAVIER = nnx.initializers.variance_scaling(2**-5, "fan_avg", "uniform")


# --------------------------------------------------------------------------- #
#  Inference cache for streaming (incremental) decode.
#
#  Linear attention's headline property: the entire history collapses into a
#  FIXED-SIZE recurrent state S [B,Hv,dk,dv] — it does NOT grow with sequence
#  length (contrast a softmax KV-cache). To decode token-by-token we just carry S
#  across calls.  The short causal conv ALSO has a (kernel_size)-wide receptive
#  field, so we must additionally cache its last (kernel_size-1) inputs — otherwise
#  the first streamed tokens would see wrong, zero-padded context.  That is the
#  WHOLE state of a GDN-2 layer; both pieces are fixed-size.
# --------------------------------------------------------------------------- #
class GDN2Cache(NamedTuple):
    recurrent_state: jax.Array  # [B, Hv, dk, dv]  the gated-delta-rule memory S
    q_conv: jax.Array  # [B, conv_size-1, H*dk]   last inputs to the q short-conv
    k_conv: jax.Array  # [B, conv_size-1, H*dk]   last inputs to the k short-conv
    v_conv: jax.Array  # [B, conv_size-1, Hv*dv]  last inputs to the v short-conv


class RMSNorm(nnx.Module):
    """Plain RMSNorm used for the pre-norms around mixer / channel-mixer."""

    def __init__(self, dim: int, *, eps: float = 1e-5, rngs: nnx.Rngs):
        self.eps = eps
        self.weight = nnx.Param(jnp.ones((dim,)))

    def __call__(self, x):
        xf = x.astype(F32)
        rms = jax.lax.rsqrt(jnp.mean(xf * xf, axis=-1, keepdims=True) + self.eps)
        return (xf * rms).astype(x.dtype) * self.weight.value


class LowRankLinear(nnx.Module):
    """y = W_up(W_down x). The W↑(W↓·) factorization Kimi Linear uses for its
    output gate and decay, kept at rank = head dim for parameter parity."""

    def __init__(
        self,
        in_features: int,
        rank: int,
        out_features: int,
        *,
        use_bias: bool = False,
        rngs: nnx.Rngs,
    ):
        self.down = nnx.Linear(
            in_features, rank, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        self.up = nnx.Linear(
            rank, out_features, use_bias=use_bias, kernel_init=_XAVIER, rngs=rngs
        )

    def __call__(self, x):
        return self.up(self.down(x))


class GatedRMSNorm(nnx.Module):
    """Head-wise RMSNorm of the recurrent output, gated by a LOW-RANK SIGMOID gate.

    Implements Kimi Linear Eq. 10's output stage:

        Sigmoid(W↑g W↓g x) ⊙ RMSNorm(O)

    Two corrections vs the GDN-2 paper block used in your gdn2_layer:
      (1) sigmoid, not SiLU/swish  — Kimi Linear's ablation found the swish output
          gate (GDN's choice) performs substantially worse than sigmoid, and they
          adopt sigmoid across all experiments including their GDN hybrid baseline.
      (2) low-rank gate (W↑ W↓), rank = head dim — Kimi Linear factorizes the gate
          "to ensure a fair parameter comparison" against the full-attention baseline.

    The gate is produced INSIDE the norm from the block input x, so the call site in
    your layer collapses to `O = self.o_norm(O_heads, x)` (no separate gate_proj).
    """

    def __init__(
        self,
        head_dim: int,
        d_model: int,
        inner_dim: int,
        gate_rank: int,
        *,
        eps: float = 1e-5,
        rngs: nnx.Rngs,
    ):
        self.eps = eps
        self.head_dim = head_dim  # dv, the axis RMSNorm normalizes over (head-wise)
        self.inner_dim = inner_dim  # Hv * dv, the full token-mixer output width
        self.weight = nnx.Param(jnp.ones((head_dim,)))
        self.gate = LowRankLinear(
            d_model, gate_rank, inner_dim, use_bias=False, rngs=rngs
        )

    def __call__(self, O_heads, x):
        """O_heads: [B, L, Hv, dv]   x: [B, L, d_model]  ->  [B, L, Hv*dv]."""
        B, L, Hv, dv = O_heads.shape
        o = O_heads.astype(F32)
        rms = jax.lax.rsqrt(jnp.mean(o * o, axis=-1, keepdims=True) + self.eps)
        o = o * rms * self.weight.value  # head-wise RMSNorm
        g = jax.nn.sigmoid(self.gate(x).astype(F32))  # low-rank SIGMOID gate
        g = g.reshape(B, L, Hv, dv)
        return (o * g).reshape(B, L, Hv * dv)


class ShortConv(nnx.Module):
    """Causal depthwise 1-D convolution — the 'Conv' boxes in Fig. 1 (Sec. 3.5).

    The paper says only "short causal convolution"; the kernel width (default 4)
    is an implementation choice, as in the Mamba/GatedDeltaNet lineage.
    """

    def __init__(self, channels: int, kernel_size: int = 4, *, rngs: nnx.Rngs):
        self.channels = channels
        self.kernel_size = kernel_size
        key = rngs.params()
        w = jax.random.normal(key, (channels, 1, kernel_size)) * (kernel_size**-0.5)
        self.weight = nnx.Param(w)
        self.bias = nnx.Param(jnp.zeros((channels,)))

    def _apply(
        self, x: jax.Array, conv_state: jax.Array | None
    ) -> tuple[jax.Array, jax.Array]:
        """Shared conv core. `conv_state` is the previous (kernel_size-1) inputs used
        as left context, or None on the full/training path (pad with zeros == the
        causal left-pad). Returns (y: [B, L, C], new_state: [B, kernel_size-1, C])."""
        B, L, C = x.shape
        kc = self.kernel_size - 1
        left = jnp.zeros((B, kc, C), x.dtype) if conv_state is None else conv_state
        xc = jnp.concatenate([left, x], axis=1)  # [B, kc+L, C]  prepend left context
        new_state = xc[
            :, xc.shape[1] - kc :, :
        ]  # last kc inputs -> next step's context
        xt = jnp.transpose(xc, (0, 2, 1))  # [B, C, kc+L]
        y = jax.lax.conv_general_dilated(
            xt,
            self.weight.value,
            window_strides=(1,),
            padding="VALID",  # out length (kc+L)-(kc+1)+1 = L; output t sees inputs t-kc..t
            feature_group_count=self.channels,  # depthwise: one filter per channel
            dimension_numbers=("NCW", "OIW", "NCW"),
        )
        y = y + self.bias.value[None, :, None]
        return jnp.transpose(y, (0, 2, 1)), new_state  # [B, L, C]

    def __call__(self, x):  # full-sequence (training) path; left context = zeros
        return self._apply(x, None)[0]

    def step(self, x, conv_state):  # streaming path; carry the left context in/out
        return self._apply(x, conv_state)


class GatedDeltaNet2(nnx.Module):
    """Gated DeltaNet-2 recurrent token mixer (Fig. 1 right; Sec. 3.5 / App. C.1)."""

    def __init__(
        self,
        d_model: int,
        num_heads: int = 16,  # H key heads; App. E.1 uses H=16 at 1.3B
        head_k_dim: int = 128,  # d_k; App. E.1 uses 128
        head_v_dim: int = 128,  # d_v; App. E.1 uses 128
        num_v_heads: int | None = None,  # H_v for GQA; defaults to H (App. C.1)
        chunk_size: int = 64,  # C; App. C.2 fixes C = 64
        conv_size: int = 4,
        expanded_erase: bool = False,  # erase gate in [0,2] (neg-eigenvalue variant; Sec. 3.1, App. C.1)
        *,
        rngs: nnx.Rngs,
    ):
        self.d_model = d_model
        self.H = num_heads
        self.Hv = num_v_heads or num_heads
        assert self.Hv % self.H == 0, "num_v_heads must be a multiple of num_heads"
        self.group = self.Hv // self.H  # G, value-head group size (App. C.1)
        self.dk = head_k_dim
        self.dv = head_v_dim
        self.chunk_size = chunk_size
        self.conv_size = conv_size  # kernel width; sizes the streaming conv cache
        self.expanded_erase = expanded_erase

        # App. C.1 projection shapes: erase/key side -> H·d_k, write/value side -> H_v·d_v.
        k_proj_dim = self.H * self.dk  # q, k, b live on the key-head axis
        v_proj_dim = self.Hv * self.dv  # v, w live on the value-head axis

        # Linear projections feeding the SiLU/conv paths (Sec. 3.5; Fig. 1 'Linear' boxes).
        self.q_proj = nnx.Linear(
            d_model, k_proj_dim, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        self.k_proj = nnx.Linear(
            d_model, k_proj_dim, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        self.v_proj = nnx.Linear(
            d_model, v_proj_dim, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        self.b_proj = nnx.Linear(
            d_model, k_proj_dim, use_bias=True, kernel_init=_XAVIER, rngs=rngs
        )  # Proj_b, Eq. 85: b = σ(Proj_b x)
        self.w_proj = nnx.Linear(
            d_model, v_proj_dim, use_bias=True, kernel_init=_XAVIER, rngs=rngs
        )  # Proj_w, Eq. 85: w = σ(Proj_w x)
        self.f_proj = LowRankLinear(
            d_model, self.dk, k_proj_dim, use_bias=True, rngs=rngs
        )  # Proj_f, Eq. 86 (log-decay)

        # Short causal convs on q, k, v (App. C.1: "short-convolutional projections for q, k, v").
        self.q_conv = ShortConv(k_proj_dim, conv_size, rngs=rngs)
        self.k_conv = ShortConv(k_proj_dim, conv_size, rngs=rngs)
        self.v_conv = ShortConv(v_proj_dim, conv_size, rngs=rngs)

        # Log-decay parameters (Eq. 12 / 86; App. C.1).
        #   DEVIATION (1): paper stores 'a' per key HEAD (shape [H]) broadcast over d_k.
        #   Here A_log is [H, d_k] (per head AND per channel) — a strict generalization.
        self.A_log = nnx.Param(
            jnp.zeros((self.H, self.dk))
        )  # 'a' in -exp(a)·softplus(·)
        # App. C.1: bias δ is stored per key channel -> shape [H·d_k]. Eq. 86 adds it pre-softplus.
        #   Init negative (not the paper's value) so per-token decay starts mild (α≈1),
        #   keeping cumulative decay / γ^{-1} in a safe fp32 range (cf. App. D.1).
        self.dt_bias = nnx.Param(jnp.full((self.H * self.dk,), -4.0))  # δ

        # Output gate + gated RMSNorm + output projection (Sec. 3.5 / App. D.5).
        self.o_norm = GatedRMSNorm(
            head_dim=self.dv,
            d_model=d_model,
            inner_dim=self.Hv * self.dv,
            gate_rank=self.dv,
            rngs=rngs,
        )
        self.o_proj = nnx.Linear(
            v_proj_dim, d_model, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )  # back to d_model
        # App. D.5: every Linear kernel above uses Xavier-uniform init, gain 2^{-2.5}
        # (_XAVIER); biases are zero except the decay bias δ (-4, for fp32 safety).

    def _split_k(self, x: jax.Array, B: int, L: int) -> jax.Array:
        # Head reshaping for key-side tensors (App. C.1: "followed by head reshaping").
        return x.reshape(B, L, self.H, self.dk).transpose(0, 2, 1, 3)  # [B,H,L,dk]

    def _split_v(self, x: jax.Array, B: int, L: int) -> jax.Array:
        # Head reshaping for value-side tensors.
        return x.reshape(B, L, self.Hv, self.dv).transpose(0, 2, 1, 3)  # [B,Hv,L,dv]

    def _project(
        self, x: jax.Array, conv_states: tuple[jax.Array, jax.Array, jax.Array] | None
    ) -> tuple[
        jax.Array,
        jax.Array,
        jax.Array,
        jax.Array,
        jax.Array,
        jax.Array,
        tuple[jax.Array, jax.Array, jax.Array] | None,
    ]:
        """Shared front-end used by BOTH the training and streaming paths:
        Linear -> ShortConv -> SiLU -> head split -> L2 norm, plus the log-decay g
        and the channel-wise gates b, w.  `conv_states` is None on the full/training
        path, or a (q, k, v) tuple of conv caches when streaming.  Returns
        (q, k, v, g, b, w) on the value-head (Hv) axis and the updated conv states
        (or None)."""
        B, L, _ = x.shape

        # q,k,v paths: Linear -> ShortConv -> SiLU (Sec. 3.5; Fig. 1 caption).
        if conv_states is None:  # full/training: conv pads with zeros (causal)
            q = self.q_conv(self.q_proj(x))
            k = self.k_conv(self.k_proj(x))
            v = self.v_conv(self.v_proj(x))
            new_conv = None
        else:  # streaming: conv uses the cached left context and returns a new one
            qcs, kcs, vcs = conv_states
            q, qcs = self.q_conv.step(self.q_proj(x), qcs)
            k, kcs = self.k_conv.step(self.k_proj(x), kcs)
            v, vcs = self.v_conv.step(self.v_proj(x), vcs)
            new_conv = (qcs, kcs, vcs)
        q, k, v = jax.nn.silu(q), jax.nn.silu(k), jax.nn.silu(v)

        q = self._split_k(q, B, L)
        k = self._split_k(k, B, L)
        v = self._split_v(v, B, L)

        # L2-normalize q, k per head (Sec. 3.5 "L2 normalization applied to q_t and k_t"; App. D.2).
        q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
        k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)

        # Log-decay branch, computed in fp32 outside the kernel (Eq. 12 / 86; App. C.1 / D.1).
        #   g_t = -exp(a) ⊙ softplus(Proj_f(x_t) + δ),  then α_t = exp(g_t) inside the core.
        f = self.f_proj(x).astype(jnp.float32) + self.dt_bias.value.astype(
            jnp.float32
        )  # Proj_f(x)+δ
        f = self._split_k(f, B, L)
        a = jnp.exp(self.A_log.value.astype(jnp.float32))[
            None, :, None, :
        ]  # exp(a); [1,H,1,dk]
        g = -a * jax.nn.softplus(f)  # [B,H,L,dk] ≤ 0  (Eq. 86)

        # Channel-wise gates (Eq. 11 / 85).
        b = jax.nn.sigmoid(self.b_proj(x))  # b = σ(Proj_b x) ∈ [0,1]^{d_k}
        b = self._split_k(b, B, L)
        if self.expanded_erase:
            b = 2.0 * b  # neg-eigenvalue variant: scale ONLY b to [0,2] (Sec. 3.1)
        w = jax.nn.sigmoid(self.w_proj(x))  # w = σ(Proj_w x) ∈ [0,1]^{d_v}
        w = self._split_v(w, B, L)

        # GQA: repeat key-side tensors across value-head groups (Sec. 3.5 / App. C.1).
        #   q, k, g, b are repeated; v, w already live on the value-head axis.
        if self.group > 1:

            def rep(t: jax.Array) -> jax.Array:
                return jnp.repeat(t, self.group, axis=1)

            q, k, g, b = rep(q), rep(k), rep(g), rep(b)

        return q, k, v, g, b, w, new_conv

    def _output(self, o: jax.Array, x: jax.Array) -> jax.Array:
        """Gated RMSNorm + output projection (Sec. 3.5 / App. D.5). o: [B,Hv,L,dv]."""
        o = o.transpose(0, 2, 1, 3)  # [B,Hv,L,dv] -> [B,L,Hv,dv]
        o = self.o_norm(o, x)  # low-rank sigmoid gate computed inside, from x
        return self.o_proj(o.astype(x.dtype))  # project back to d_model

    def __call__(
        self, x: jax.Array, initial_state: jax.Array | None = None
    ) -> jax.Array:
        """Full-sequence (training) forward via the CHUNKWISE parallel core.
        x: [B, L, d_model]. Returns (out: [B, L, d_model], final_state: [B,Hv,dk,dv])."""
        B, L, _ = x.shape
        q, k, v, g, b, w, _ = self._project(x, conv_states=None)
        if initial_state is None:
            initial_state = jnp.zeros((B, self.Hv, self.dk, self.dv), jnp.float32)
        # Gated Delta Rule-2 chunkwise core (Eq. 10); forms cumsum γ internally (Eq. 30).
        o, _final_state = chunkwise_gated_delta_rule_2(
            q, k, v, g, b, w, initial_state, chunk_size=self.chunk_size
        )
        return self._output(o, x)

    # ----------------------------------------------------------------------- #
    #  Streaming / inference.  Same math, but via the RECURRENT core, which works
    #  for ANY length (no chunk-size divisibility constraint) and naturally threads
    #  the fixed-size state in -> out.  One method serves both phases of decoding:
    #     prefill: out, cache = layer.step(prompt, layer.init_cache(B, ...))
    #     decode : out, cache = layer.step(one_token, cache)   # repeat
    # ----------------------------------------------------------------------- #
    def init_cache(
        self, batch_size: int, max_len: int | None = None, dtype=jnp.float32
    ) -> GDN2Cache:
        """Empty streaming cache. `max_len` is accepted for a uniform interface with
        the MLA cache but UNUSED here — the GDN-2 state is fixed-size, independent of
        sequence length (the point of linear attention)."""
        kc = self.conv_size - 1
        return GDN2Cache(
            recurrent_state=jnp.zeros(
                (batch_size, self.Hv, self.dk, self.dv), jnp.float32
            ),
            q_conv=jnp.zeros((batch_size, kc, self.H * self.dk), dtype),
            k_conv=jnp.zeros((batch_size, kc, self.H * self.dk), dtype),
            v_conv=jnp.zeros((batch_size, kc, self.Hv * self.dv), dtype),
        )

    def step(self, x: jax.Array, cache: GDN2Cache) -> tuple[jax.Array, GDN2Cache]:
        """Streaming forward. x: [B, L, d_model] (L>=1). Returns (out, new_cache)."""
        q, k, v, g, b, w, new_conv = self._project(
            x, conv_states=(cache.q_conv, cache.k_conv, cache.v_conv)
        )
        # We passed real conv_states, so _project always returns updated ones here
        # (it only returns None on the full-sequence/training path) — assert narrows
        # the tuple|None type for the checker and documents the invariant.
        assert new_conv is not None
        qcs, kcs, vcs = new_conv
        # Recurrent core: token-by-token, threading S_in -> S_out (Eq. 9 / 29).
        o, new_state = recurrent_gated_delta_rule_2(
            q, k, v, g, b, w, cache.recurrent_state
        )
        return self._output(o, x), GDN2Cache(new_state, qcs, kcs, vcs)


In [ ]:
# App. D.5: Xavier-uniform init with gain 2^{-2.5} (variance_scaling scale = gain² =
# 2^{-5}), replacing Flax NNX's default Linear kernel init. Biases stay at zero.
_XAVIER = nnx.initializers.variance_scaling(2**-5, "fan_avg", "uniform")


class MLACache(NamedTuple):
    """Streaming KV cache for the MLA layer. Thanks to MLA we cache only the small
    COMPRESSED latent `l_kv` (one latent serves as BOTH K and V — see below), in a
    preallocated [B, max_len, Hkv*Dh] buffer written at position `pos`. Unlike GDN-2's
    fixed-size state, this GROWS with context: these full-attention layers are exactly
    the ones that pay the long-context KV-cache cost in the hybrid (3:1 keeps them few)."""

    l_kv: jax.Array  # [B, max_len, num_kv_heads*head_dim]  preallocated latent buffer
    pos: jax.Array  # scalar int32: number of filled positions so far


class GroupedQueryLatentAttention(nnx.Module):
    """Grouped-Query attention over a low-rank KV *latent*, in MLA "absorbed" form.

    This is NoPE (no rotary embeddings) Multi-head Latent Attention written in its
    matrix-absorbed form, fused with GQA-style KV-head sharing. Each of the three
    projections folds together two of the usual MLA matrices:

        w_q_uk : W_Q  . W_UK   -> queries are produced *directly* in the
                                  compressed K space, so they can dot against the
                                  latent without an explicit key up-projection.
        w_dkv  : W_DKV          -> down-projects x to the shared KV latent (c_kv).
        w_uv_o : W_UV . W_O     -> up-projects the value latent and applies the
                                  output projection in a single matmul.

    Key consequence: because there is no RoPE, W_UK and W_UV can be absorbed away
    *exactly*, and in the compressed latent space the keys and the values are the
    same tensor. That is why a single `l_kv` plays the role of BOTH K and V below.

    Note: `head_dim` here is the per-head latent (rank) dimension, not a
    conventional attention head width.
    """

    def __init__(
        self,
        embed_dim: int,
        num_q_heads: int,
        num_kv_heads: int,
        head_dim: int,
        dropout_rate: float,
        seq_length: int,
        rngs: nnx.Rngs,
    ):
        # GQA constraint: every KV (latent) head must serve a whole number of
        # query heads, so that `repeat` below tiles the latent evenly.
        if num_q_heads % num_kv_heads != 0:
            raise ValueError(
                f"num_q_heads ({num_q_heads}) must be divisible by num_kv_heads ({num_kv_heads})."
            )

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim

        # How many query heads share each KV/latent head (the GQA group size).
        self.group_size = num_q_heads // num_kv_heads

        d_q = num_q_heads * head_dim  # total width of the query projection
        d_kv = num_kv_heads * head_dim  # total width of the (shared) KV latent

        # W_Q . W_UK absorbed: x -> queries already living in the latent K space.
        self.w_q_uk = nnx.Linear(
            embed_dim, d_q, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        # W_DKV: x -> low-rank KV latent c_kv (one latent per KV head).
        self.w_dkv = nnx.Linear(
            embed_dim, d_kv, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        # W_UV . W_O absorbed: value-latent -> up-projected, output-projected.
        self.w_uv_o = nnx.Linear(
            d_q, embed_dim, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )

        # Lower-triangular causal mask (True = keep), built once at the
        # construction seq_length and sliced at call time, so it also covers any
        # shorter sequence. The diagonal is included, guaranteeing at least one
        # unmasked key per row (so the -inf masking below cannot NaN).
        #
        # Wrapped in nnx.Variable so NNX treats it as a proper *state leaf* (data)
        # rather than a static attribute. It is a plain Variable, not an nnx.Param,
        # so optimizers that filter on Param leave it untouched -- correct for a
        # constant. It is still carried in the module state (moved/checkpointed
        # with the model). Indexing it (below) returns the underlying array.
        self.causal_mask = nnx.Variable(
            jnp.tril(jnp.ones((seq_length, seq_length), dtype=bool))
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        # x: (B, T, embed_dim)
        batch_size, seq_length, _ = x.shape

        # --- Queries (already in the compressed K space via the absorbed W_UK) ---
        q_latent = self.w_q_uk(x)  # (B, T, num_q_heads * head_dim)

        # Split the flat projection into per-head latent vectors.
        q_reshaped = q_latent.reshape(
            batch_size, seq_length, self.num_q_heads, self.head_dim
        )  # (B, T, Hq, Dh)

        # Move the head axis next to batch for batched matmuls: (B, Hq, T, Dh)
        q_heads = q_reshaped.swapaxes(1, 2)

        # --- Shared KV latent (serves as both keys and values) ---
        l_kv = self.w_dkv(x)  # (B, T, num_kv_heads * head_dim)

        l_kv_reshaped = l_kv.reshape(
            batch_size, seq_length, self.num_kv_heads, self.head_dim
        )  # (B, T, Hkv, Dh)

        l_kv_heads = l_kv_reshaped.swapaxes(1, 2)  # (B, Hkv, T, Dh)

        # GQA tiling: repeat each latent head `group_size` times so it lines up
        # with the query heads. `repeat` interleaves, so KV head i feeds query
        # heads [i*group_size : (i+1)*group_size]. Result: (B, Hq, T, Dh).
        # (This materializes the full Hq KV stack; broadcasting would save memory
        # but materializing keeps the einsums simple.)
        l_kv_repeated = l_kv_heads.repeat(self.group_size, axis=1)

        # --- Attention scores: Q . K^T, contracting the latent feature dim `d` ---
        # 'd' is shared (contracted); 'k' indexes key/latent positions (kept).
        qk_t = jnp.einsum("bhqd, bhkd -> bhqk", q_heads, l_kv_repeated)  # (B, Hq, T, T)

        # Scale by sqrt of the latent per-head dim.
        scaled_logits = qk_t / jnp.sqrt(self.head_dim)

        # Apply causal mask: future positions -> -inf so they vanish under softmax.
        # Indexing the nnx.Variable yields the raw bool array. Safe to use -inf
        # here because the diagonal is always kept (no fully-masked rows).
        scaled_logits = jnp.where(
            self.causal_mask[None, None, :seq_length, :seq_length],
            scaled_logits,
            -jnp.inf,
        )

        # Softmax over the key axis -> per-query attention distribution.
        a = jax.nn.softmax(scaled_logits, axis=-1)  # (B, Hq, T, T)

        # --- Weighted sum of value-latents ---
        # 'k' is shared between the weights and the value positions, so it is the
        # contracted axis (the actual attention sum); 'd' is the kept feature dim.
        # Because keys and values are the same latent, l_kv_repeated reappears here.
        weighted_heads = jnp.einsum(
            "bhqk, bhkd -> bhqd", a, l_kv_repeated
        )  # (B, Hq, T, Dh)

        # Move head axis back and flatten heads: (B, T, Hq, Dh) -> (B, T, Hq*Dh)
        weighted_reshaped = weighted_heads.swapaxes(1, 2)
        weighted_latents = weighted_reshaped.reshape(
            batch_size, seq_length, self.num_q_heads * self.head_dim
        )

        # Absorbed W_UV . W_O: up-project the value latent and output-project.
        output = self.w_uv_o(weighted_latents)  # (B, T, embed_dim)

        return output

    # ----------------------------------------------------------------------- #
    #  Streaming / inference.  Same softmax attention, but the KV latents of past
    #  positions are read from a preallocated cache instead of recomputed, and the
    #  new positions are written into it.  Use it for prefill (L = prompt length)
    #  and per-token decode (L = 1) alike.
    # ----------------------------------------------------------------------- #
    def init_cache(
        self, batch_size: int, max_len: int, dtype=jnp.float32
    ) -> MLACache:
        """Empty cache: a zeroed latent buffer of capacity `max_len`, position 0."""
        d_kv = self.num_kv_heads * self.head_dim
        return MLACache(
            l_kv=jnp.zeros((batch_size, max_len, d_kv), dtype),
            pos=jnp.array(0, jnp.int32),
        )

    def step(self, x: jax.Array, cache: MLACache) -> tuple[jax.Array, MLACache]:
        """Streaming attention. x: [B, L, embed_dim] (L>=1). Returns (out, new_cache)."""
        B, L, _ = x.shape
        max_len = cache.l_kv.shape[1]

        # Queries for the new positions (already in the compressed K space via W_UK).
        q_heads = (
            self.w_q_uk(x)
            .reshape(B, L, self.num_q_heads, self.head_dim)
            .swapaxes(1, 2)
        )  # (B, Hq, L, Dh)

        # New latents -> write them into the cache buffer at the current position.
        l_new = self.w_dkv(x)  # (B, L, Hkv*Dh)
        l_kv = jax.lax.dynamic_update_slice(
            cache.l_kv, l_new.astype(cache.l_kv.dtype), (0, cache.pos, 0)
        )
        new_pos = cache.pos + L

        l_kv_heads = l_kv.reshape(
            B, max_len, self.num_kv_heads, self.head_dim
        ).swapaxes(1, 2)  # (B, Hkv, max_len, Dh)
        l_kv_rep = l_kv_heads.repeat(self.group_size, axis=1)  # (B, Hq, max_len, Dh)

        # Scores: the L new queries attend over all max_len cached slots.
        logits = jnp.einsum(
            "bhqd, bhkd -> bhqk", q_heads, l_kv_rep
        ) / jnp.sqrt(self.head_dim)

        # Causal mask offset by the cache position: query i sits at absolute position
        # pos+i and may attend to slot j iff j <= pos+i.  This also masks the not-yet-
        # filled slots (j >= pos+L > pos+i), so no separate validity mask is needed.
        q_pos = cache.pos + jnp.arange(L)  # (L,)
        k_pos = jnp.arange(max_len)  # (max_len,)
        mask = k_pos[None, :] <= q_pos[:, None]  # (L, max_len)
        logits = jnp.where(mask[None, None], logits, -jnp.inf)

        a = jax.nn.softmax(logits, axis=-1)
        weighted = jnp.einsum("bhqk, bhkd -> bhqd", a, l_kv_rep)  # (B, Hq, L, Dh)
        weighted = weighted.swapaxes(1, 2).reshape(
            B, L, self.num_q_heads * self.head_dim
        )
        return self.w_uv_o(weighted), MLACache(l_kv, new_pos)


In [ ]:
"""
Dispatched grouped-GEMM MoE channel mixer for Kimi Linear (JAX / Flax NNX).

Replaces the dense O(E) reference MoE in kimi_linear_gdn2.py with the production
pattern: permute tokens so each expert's assignments are contiguous (dispatch),
run one matmul per expert as a single grouped GEMM (`jax.lax.ragged_dot`), then
un-permute and weighted-sum (combine). No token dropping, no capacity padding.

Pipeline per forward:
    1. Route:    sigmoid affinities -> top-k experts (+ aux-loss-free bias on the
                 SELECTION only) -> normalize the k gate weights.
    2. Dispatch: build (token, expert) assignments, sort by expert id, gather the
                 hidden states into expert-contiguous order; group_sizes = per-expert
                 counts.
    3. Grouped GEMM: ragged_dot(x_sorted, W_in, group_sizes) -> SwiGLU ->
                 ragged_dot(a, W_out, group_sizes).   (gate+up fused into W_in)
    4. Combine:  scale rows by gate weight, scatter-add back to tokens (sums the
                 top-k contributions), add the always-on shared expert.

Routing follows the DeepSeek-V3 / Moonlight / Kimi lineage: sigmoid scoring,
normalized top-k weights, a shared expert, and aux-loss-free load balancing via a
per-expert selection bias updated outside the gradient (see `update_router_bias`).
Group-limited ("device-limited") routing is omitted for clarity; see note below.
"""

F32 = jnp.float32

# App. D.5: Xavier-uniform init with gain 2^{-2.5} (variance_scaling scale = gain² =
# 2^{-5}), replacing Flax NNX's default Linear kernel init. Biases stay at zero (the
# NNX default). The stacked expert weights below keep their own explicit fan-in init.
_XAVIER = nnx.initializers.variance_scaling(2**-5, "fan_avg", "uniform")


class GroupedGemmMoE(nnx.Module):
    """Token-dispatched grouped-GEMM MoE with a shared expert.

    Args:
        d_model:   model width.
        d_ff:      per-expert hidden width (SwiGLU inner dim).
        n_routed:  number of routed experts E.
        n_shared:  number of shared experts (always-on), folded into one SwiGLU.
        top_k:     experts activated per token.
        norm_topk: renormalize the top-k gate weights to sum to 1.
        routed_scale: multiply normalized gate weights (DeepSeek's routed_scaling_factor).
        bias_balancing: enable the aux-loss-free selection bias.
        aux_alpha: coefficient for the optional sequence-level load-balancing aux loss.
    """

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        n_routed: int = 256,
        n_shared: int = 1,
        top_k: int = 8,
        *,
        norm_topk: bool = True,
        routed_scale: float = 1.0,
        bias_balancing: bool = True,
        aux_alpha: float = 1e-3,
        rngs: nnx.Rngs,
    ):
        self.d_model = d_model
        self.d_ff = d_ff
        self.E = n_routed
        self.top_k = top_k
        self.norm_topk = norm_topk
        self.routed_scale = routed_scale
        self.bias_balancing = bias_balancing
        self.aux_alpha = aux_alpha

        self.router = nnx.Linear(
            d_model, n_routed, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )
        self.router_bias = nnx.Variable(jnp.zeros((n_routed,), F32))

        # Stacked routed-expert weights. Gate and up are fused into W_in so the
        # forward needs only TWO grouped GEMMs (W_in, W_out) instead of three.
        kin, kout = jax.random.split(rngs.params(), 2)
        self.w_in = nnx.Param(
            jax.random.normal(kin, (n_routed, d_model, 2 * d_ff), F32) * (d_model**-0.5)
        )
        self.w_out = nnx.Param(
            jax.random.normal(kout, (n_routed, d_ff, d_model), F32) * (d_ff**-0.5)
        )

        # Shared expert(s) as a single wider SwiGLU (always applied to every token).
        sg, su, sd = jax.random.split(rngs.params(), 3)
        ish = d_ff * n_shared
        self.ws_gate = nnx.Param(
            jax.random.normal(sg, (d_model, ish), F32) * (d_model**-0.5)
        )
        self.ws_up = nnx.Param(
            jax.random.normal(su, (d_model, ish), F32) * (d_model**-0.5)
        )
        self.ws_down = nnx.Param(
            jax.random.normal(sd, (ish, d_model), F32) * (ish**-0.5)
        )

    # ----------------------------------------------------------------------- #
    def _route(self, x_flat: jax.Array) -> tuple[jax.Array, jax.Array, jax.Array]:
        """x_flat: [T, d] -> (top_idx [T,k], gate [T,k], scores [T,E])."""
        logits = self.router(x_flat).astype(F32)
        scores = jax.nn.sigmoid(logits)  # affinities [T,E]

        sel = scores + self.router_bias if self.bias_balancing else scores
        sel = jax.lax.stop_gradient(sel) if self.bias_balancing else sel
        _, top_idx = jax.lax.top_k(sel, self.top_k)  # selection [T,k]

        gate = jnp.take_along_axis(scores, top_idx, axis=-1)  # gate from true scores
        if self.norm_topk:
            gate = gate / (gate.sum(-1, keepdims=True) + 1e-9)
        gate = gate * self.routed_scale
        return top_idx, gate, scores

    def _shared(self, x_flat: jax.Array) -> jax.Array:
        a = jax.nn.silu(x_flat @ self.ws_gate) * (x_flat @ self.ws_up)
        return a @ self.ws_down

    # ----------------------------------------------------------------------- #
    def __call__(self, x: jax.Array) -> tuple[jax.Array, dict[str, jax.Array]]:
        B, L, d = x.shape
        T = B * L
        k = self.top_k
        xf = x.reshape(T, d)
        cdtype = x.dtype

        top_idx, gate, scores = self._route(xf)

        # ---- dispatch: flatten assignments and sort by expert id ----
        flat_e = top_idx.reshape(T * k).astype(jnp.int32)  # expert per assignment
        flat_tok = jnp.repeat(jnp.arange(T, dtype=jnp.int32), k)  # token per assignment
        flat_w = gate.reshape(T * k).astype(F32)

        order = jnp.argsort(flat_e)  # group same-expert rows
        sort_tok = flat_tok[order]
        sort_w = flat_w[order]
        group_sizes = jnp.bincount(flat_e, length=self.E)  # [E], sums to T*k

        x_sorted = xf[sort_tok].astype(cdtype)  # [M, d], M = T*k

        # ---- grouped GEMM: one matmul per expert over its contiguous rows ----
        h = jax.lax.ragged_dot(x_sorted, self.w_in.astype(cdtype), group_sizes)
        g_, u_ = jnp.split(h, 2, axis=-1)  # [M, d_ff] each
        a = jax.nn.silu(g_) * u_
        y_sorted = jax.lax.ragged_dot(
            a, self.w_out.astype(cdtype), group_sizes
        )  # [M,d]

        # ---- combine: weight, un-permute, sum top-k per token ----
        y_sorted = y_sorted.astype(F32) * sort_w[:, None]
        routed = (
            jnp.zeros((T, d), F32).at[sort_tok].add(y_sorted)
        )  # scatter-add over slots

        out = routed + self._shared(xf).astype(F32)
        out = out.reshape(B, L, d).astype(cdtype)

        # ---- diagnostics for the training loop ----
        load = group_sizes.astype(F32) / (T * k)  # fraction per expert

        # ---- aux loss ----
        # Switch/DeepSeek aux loss: E * <f_e, P_e>, P from softmax routing probs.
        probs = jax.nn.softmax(self.router(xf).astype(F32), axis=-1).mean(0)  # [E]
        aux_loss = self.aux_alpha * self.E * jnp.sum(load * probs)
        aux = {"load": load, "aux_loss": aux_loss, "group_sizes": group_sizes}
        return out, aux

    # ----------------------------------------------------------------------- #
    def dense_forward(self, x: jax.Array) -> jax.Array:
        """Reference path computing every expert densely (for tests only).
        Uses the SAME weights as __call__, so any mismatch is a dispatch/GEMM bug."""
        B, L, d = x.shape
        T = B * L
        xf = x.reshape(T, d)
        top_idx, gate, _ = self._route(xf)

        full = (
            jnp.zeros((T, self.E), F32).at[jnp.arange(T)[:, None], top_idx].add(gate)
        )  # [T,E] sparse weights
        h = jnp.einsum("td,edf->tef", xf, self.w_in)  # [T,E,2*d_ff]
        g_, u_ = jnp.split(h, 2, axis=-1)
        a = jax.nn.silu(g_) * u_
        ye = jnp.einsum("tef,efd->ted", a, self.w_out)  # [T,E,d]
        routed = jnp.einsum("te,ted->td", full, ye)
        out = routed + self._shared(xf)
        return out.reshape(B, L, d)


# --------------------------------------------------------------------------- #
def update_router_bias(
    bias: jax.Array, group_sizes: jax.Array, lr: float = 1e-3
) -> jax.Array:
    """Aux-loss-free load balancing (DeepSeek-V3 style), called in the training loop
    AFTER each step, outside the gradient:

        moe.router_bias = update_router_bias(
            moe.router_bias, aux['group_sizes'], lr)

    Nudges the selection bias up for under-loaded experts and down for over-loaded
    ones by a fixed step, driving per-expert load toward uniform without an aux loss.
    """
    load = group_sizes.astype(F32) / jnp.sum(group_sizes).astype(F32)
    target = 1.0 / bias.shape[0]
    return bias + lr * jnp.sign(target - load)


# Note on group-limited routing: Kimi K2 / DeepSeek-V3 also restrict each token to
# experts drawn from a few expert groups (device-limited routing) to bound all-to-all
# traffic. That is a routing-side refinement layered before top_k; it does not change
# the dispatch / grouped-GEMM / combine machinery above. Add it inside `_route` by
# masking `sel` to the top expert-groups per token before the final top_k.


In [ ]:
"""
Kimi Linear (GDN-2 variant) — the top-level decoder-only language model, in JAX /
Flax NNX. ANNOTATED against "Kimi Linear: An Expressive, Efficient Attention
Architecture."

WHAT KIMI LINEAR IS (paper, Sec. 3 / Fig. 2)
--------------------------------------------
A *hybrid* linear-attention transformer. Most layers use a cheap, O(L) linear-
attention token mixer (the paper's "Kimi Delta Attention", KDA); a minority use
ordinary softmax full attention (Multi-head Latent Attention, MLA). The two are
interleaved at a fixed **3:1 ratio** — three linear layers for every one full-
attention layer — which the paper finds recovers full-attention quality at a
fraction of the KV-cache and compute cost.

  • KDA layers carry positional information implicitly through their recurrence,
    so the full-attention layers need NO positional encoding. Hence the MLA layers
    here are NoPE (see multi_latent_attention/attention.py).
  • Every layer's channel mixer (FFN) is a DeepSeek-V3 / Moonlight-style MoE.

THIS FILE'S ONE DELIBERATE SUBSTITUTION
---------------------------------------
We replace KDA with **Gated DeltaNet-2** ("Decoupling Erase and Write in Linear
Attention", arXiv:2605.22791). Both are gated-delta-rule linear attentions with
fine-grained (channel-wise) gating; GDN-2's twist is a separate erase gate `b` and
write gate `w` instead of the single `beta` that KDA/GDN share. Everything else of
Kimi Linear — the 3:1 hybrid schedule, NoPE MLA, MoE FFN, pre-norm residual blocks
— is kept as in the paper. See gated_deltanet_2/layer.py for that token mixer.

BLOCK STRUCTURE (standard pre-norm transformer; Fig. 2)
-------------------------------------------------------
    x = x + TokenMixer(RMSNorm(x))     # TokenMixer = GDN-2 (linear) OR MLA (full)
    x = x + ChannelMixer(RMSNorm(x))   # ChannelMixer = MoE (or a dense SwiGLU MLP)

MODEL = Embed -> [DecoderLayer] * n_layers -> RMSNorm -> LM head.

TWO FORWARD MODES
-----------------
  • Training / full sequence:  model(input_ids)  — parallel, GDN-2 via its chunkwise
    core, MLA via a full causal-attention matrix.
  • Streaming / inference:     model.step(ids, caches) and model.generate(...)  —
    reuses per-layer state across calls so each new token is O(1) work for the GDN-2
    layers (fixed-size recurrent state) and O(context) for the few MLA layers (growing
    latent cache). See GatedDeltaNet2.step / GroupedQueryLatentAttention.step.
"""

from __future__ import annotations

import dataclasses

import flax.nnx as nnx
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike

# Reuse the building blocks already implemented and verified in this repo.
from gated_deltanet_2.layer import GatedDeltaNet2, RMSNorm
from multi_latent_attention.attention import GroupedQueryLatentAttention
from multi_latent_attention.moe import GroupedGemmMoE

# App. D.5: Xavier-uniform init with gain 2^{-2.5} (variance_scaling scale = gain² =
# 2^{-5}) for the embedding and LM head, replacing Flax NNX's defaults. The (small)
# embedding scale this produces is fine — RMSNorm renormalizes the residual stream.
_XAVIER = nnx.initializers.variance_scaling(2**-5, "fan_avg", "uniform")


# --------------------------------------------------------------------------- #
#  Configuration
#
#  Defaults are deliberately TINY so the whole model trains on a laptop CPU. The
#  paper's 48B-A3B numbers are quoted in comments for reference; only the *ratios*
#  and structure matter for understanding — scale up by raising the dims/layers.
# --------------------------------------------------------------------------- #
@dataclasses.dataclass
class KimiLinearConfig:
    vocab_size: int = 256  # paper: 160k; tiny here (byte-level demo)
    d_model: int = 256  # model width  (paper 1.3B: 2048)
    n_layers: int = 8  # depth        (paper 1.3B: 27)

    # --- Hybrid schedule: which layers are FULL attention (MLA) vs linear (GDN-2) ---
    # full_attn_period = 4 places one MLA layer every 4th layer (indices 3, 7, ...),
    # i.e. a 3:1 linear:full ratio — exactly Kimi Linear's hybrid recipe (Sec. 3.2).
    full_attn_period: int = 4

    # --- GDN-2 token mixer (the KDA replacement) — see gated_deltanet_2/layer.py ---
    gdn_num_heads: int = 4  # H key/query heads   (paper 1.3B: 16)
    gdn_head_k_dim: int = 64  # d_k                 (paper: 128)
    gdn_head_v_dim: int = 64  # d_v                 (paper: 128)
    gdn_num_v_heads: int | None = None  # H_v for GQA value heads; None -> = num_heads
    gdn_chunk_size: int = 64  # chunkwise block size C (paper App.: 64).
    #   NOTE: the GDN-2 chunkwise core requires every fed sequence length to be a
    #   multiple of this C (it reshapes L into L/C chunks). Keep seq_len % C == 0.
    gdn_conv_size: int = 4  # short-conv kernel width
    gdn_expanded_erase: bool = False  # erase gate in [0,2] (neg-eigenvalue variant)

    # --- MLA full-attention layers (NoPE) — see multi_latent_attention/attention.py ---
    mla_num_q_heads: int = 8  # query heads
    mla_num_kv_heads: int = 2  # KV/latent heads (GQA); q_heads must be a multiple
    mla_head_dim: int = 64  # per-head latent (rank) width
    max_seq_len: int = 512  # builds the causal mask; cap on trainable length

    # --- Channel mixer (FFN) ---
    moe_d_ff: int = 512  # per-expert hidden width (paper: 1408 at 1.3B)
    moe_n_routed: int = 8  # number of routed experts E (paper: 256)
    moe_n_shared: int = 1  # always-on shared experts
    moe_top_k: int = 2  # experts activated per token (paper: 8)
    mlp_d_ff: int = 768  # hidden width of the dense MLP fallback

    rms_eps: float = 1e-5


# --------------------------------------------------------------------------- #
#  One decoder block: pre-norm token mixer + pre-norm channel mixer, both residual.
#
#  The ONLY thing that varies across layers is the token mixer: GDN-2 (linear) on
#  most layers, MLA (full attention) on the 3:1 schedule. The channel mixer (MoE or
#  dense MLP) is the same kind on every layer — this matches Kimi Linear, where the
#  hybrid is in the *attention*, not the FFN.
# --------------------------------------------------------------------------- #
class DecoderLayer(nnx.Module):
    def __init__(self, cfg: KimiLinearConfig, layer_idx: int, *, rngs: nnx.Rngs):
        # 3:1 schedule: this layer is full-attention iff it is the last of its period.
        self.is_full_attn = (layer_idx + 1) % cfg.full_attn_period == 0

        # Pre-norm before the token mixer (Fig. 2). RMSNorm reused from the GDN-2 layer.
        self.norm1 = RMSNorm(cfg.d_model, eps=cfg.rms_eps, rngs=rngs)

        if self.is_full_attn:
            # Full attention: NoPE Multi-head Latent Attention (absorbed/GQA form).
            self.token_mixer = GroupedQueryLatentAttention(
                embed_dim=cfg.d_model,
                num_q_heads=cfg.mla_num_q_heads,
                num_kv_heads=cfg.mla_num_kv_heads,
                head_dim=cfg.mla_head_dim,
                dropout_rate=0.0,
                seq_length=cfg.max_seq_len,
                rngs=rngs,
            )
        else:
            # Linear attention: Gated DeltaNet-2 (the KDA substitute).
            self.token_mixer = GatedDeltaNet2(
                d_model=cfg.d_model,
                num_heads=cfg.gdn_num_heads,
                head_k_dim=cfg.gdn_head_k_dim,
                head_v_dim=cfg.gdn_head_v_dim,
                num_v_heads=cfg.gdn_num_v_heads,
                chunk_size=cfg.gdn_chunk_size,
                conv_size=cfg.gdn_conv_size,
                expanded_erase=cfg.gdn_expanded_erase,
                rngs=rngs,
            )

        # Pre-norm before the channel mixer.
        self.norm2 = RMSNorm(cfg.d_model, eps=cfg.rms_eps, rngs=rngs)

        # Channel mixer: MoE
        self.channel_mixer = GroupedGemmMoE(
            d_model=cfg.d_model,
            d_ff=cfg.moe_d_ff,
            n_routed=cfg.moe_n_routed,
            n_shared=cfg.moe_n_shared,
            top_k=cfg.moe_top_k,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> tuple[jax.Array, dict[str, jax.Array]]:
        """x: [B, L, d_model] -> (x, aux_or_None).

        `aux` carries the MoE load-balancing diagnostics the training loop
        needs (aux loss + per-expert token counts for the router-bias update).
        """
        # --- token mixing (residual, pre-norm) ---
        h = self.norm1(x)
        h = self.token_mixer(h)
        x = x + h

        # --- channel mixing (residual, pre-norm) ---
        y = self.norm2(x)
        m, aux = self.channel_mixer(y)
        x = x + m
        return x, aux

    def init_cache(self, batch_size: int, max_len: int, dtype=jnp.float32):
        """Per-layer streaming cache: a GDN2Cache (linear layer) or MLACache (MLA)."""
        return self.token_mixer.init_cache(batch_size, max_len, dtype)

    def step(self, x: jax.Array, cache):
        """Streaming forward for one block. x: [B, L, d_model] -> (x, new_cache).
        Only the token mixer is stateful; the channel mixer (MoE/MLP) is position-wise,
        so it needs no cache."""
        h = self.norm1(x)
        h, new_cache = self.token_mixer.step(
            h, cache
        )  # GDN-2 and MLA both expose .step
        x = x + h
        y = self.norm2(x)
        m, _aux = self.channel_mixer(y)
        x = x + m
        return x, new_cache


# --------------------------------------------------------------------------- #
#  The full model.
# --------------------------------------------------------------------------- #
class KimiLinear(nnx.Module):
    """Decoder-only Kimi Linear LM with a GDN-2 linear-attention backbone."""

    def __init__(self, cfg: KimiLinearConfig, *, rngs: nnx.Rngs):
        self.cfg = cfg
        # Token embedding table.
        self.embed = nnx.Embed(
            cfg.vocab_size, cfg.d_model, embedding_init=_XAVIER, rngs=rngs
        )
        # Stack of decoder blocks. NOTE: in Flax NNX a plain Python list of submodules
        # is not tracked as state — it must be wrapped in nnx.List(...).
        self.layers = nnx.List(
            [DecoderLayer(cfg, i, rngs=rngs) for i in range(cfg.n_layers)]
        )
        # Final pre-head norm + untied LM head (Moonlight/DeepSeek do not tie weights;
        # to tie, drop lm_head and use `x @ self.embed.embedding.value.T` instead).
        self.norm_f = RMSNorm(cfg.d_model, eps=cfg.rms_eps, rngs=rngs)
        self.lm_head = nnx.Linear(
            cfg.d_model, cfg.vocab_size, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )

    def __call__(
        self, input_ids: jax.Array, return_aux: bool = False
    ) -> tuple[jax.Array, dict[str, ArrayLike]]:
        """input_ids: int[B, L] -> logits[B, L, vocab]  (or (logits, aux) if return_aux).

        aux = {"aux_loss": scalar summed over MoE layers,
               "group_sizes": list of per-expert token counts, one entry per layer}.
        """
        x = self.embed(input_ids)  # [B, L, d_model]

        aux_loss: ArrayLike = 0.0
        group_sizes: list[
            ArrayLike
        ] = []  # one [E] vector per MoE layer, in layer order
        for layer in self.layers:
            x, aux = layer(x)

            aux_loss = aux_loss + aux["aux_loss"]
            group_sizes.append(aux["group_sizes"])

        x = self.norm_f(x)
        logits = self.lm_head(x)  # [B, L, vocab]

        return logits, {"aux_loss": aux_loss, "group_sizes": jnp.stack(group_sizes)}

    # ----------------------------------------------------------------------- #
    #  Streaming / inference.  Each layer carries its own cache (GDN-2: fixed-size
    #  recurrent state + conv state; MLA: growing latent cache).  Reusing them makes
    #  generation O(1) per token for the linear layers instead of re-reading history.
    # ----------------------------------------------------------------------- #
    def init_cache(
        self, batch_size: int, max_len: int | None = None, dtype=jnp.float32
    ) -> list:
        """Streaming caches for every layer. `max_len` (default cfg.max_seq_len) sizes
        the MLA latent buffers; GDN-2 layers ignore it (their state is fixed-size)."""
        max_len = max_len or self.cfg.max_seq_len
        return [layer.init_cache(batch_size, max_len, dtype) for layer in self.layers]

    def step(self, input_ids: jax.Array, caches: list) -> tuple[jax.Array, list]:
        """One streaming step. input_ids: int[B, L] (L = prompt length on prefill, or
        1 per decoded token). Returns (logits[B, L, vocab], new_caches)."""
        x = self.embed(input_ids)
        new_caches = []
        for layer, cache in zip(self.layers, caches):
            x, new_cache = layer.step(x, cache)
            new_caches.append(new_cache)
        x = self.norm_f(x)
        return self.lm_head(x), new_caches

    def generate(
        self, prompt_ids: jax.Array, max_new_tokens: int, max_len: int | None = None
    ) -> jax.Array:
        """Greedy autoregressive decode that REUSES each layer's state across steps.
        prompt_ids: int[B, P]. Returns the continuation int[B, max_new_tokens].

        Prefill consumes the whole prompt in one step (filling every layer's cache);
        each decode step then feeds back ONE token and carries the caches forward — the
        GDN-2 layers via their fixed-size recurrent state, the MLA layers via the
        growing latent cache. (Wrap `step` in nnx.jit for a fast decode loop.)"""
        B, P = prompt_ids.shape
        max_len = max_len or (P + max_new_tokens)
        caches = self.init_cache(B, max_len)
        logits, caches = self.step(prompt_ids, caches)  # prefill the prompt
        next_tok = jnp.argmax(logits[:, -1:], axis=-1)  # [B, 1] greedy
        outs = [next_tok]
        for _ in range(max_new_tokens - 1):
            logits, caches = self.step(next_tok, caches)  # decode one token
            next_tok = jnp.argmax(logits[:, -1:], axis=-1)
            outs.append(next_tok)
        return jnp.concatenate(outs, axis=1)  # [B, max_new_tokens]


def count_params(model: nnx.Module) -> int:
    """Total number of trainable parameters (sum of nnx.Param leaf sizes)."""
    return int(sum(x.size for x in jax.tree.leaves(nnx.state(model, nnx.Param))))


In [ ]:
"""
Kimi Linear (GDN-2 variant) — the top-level decoder-only language model, in JAX /
Flax NNX. ANNOTATED against "Kimi Linear: An Expressive, Efficient Attention
Architecture."

WHAT KIMI LINEAR IS (paper, Sec. 3 / Fig. 2)
--------------------------------------------
A *hybrid* linear-attention transformer. Most layers use a cheap, O(L) linear-
attention token mixer (the paper's "Kimi Delta Attention", KDA); a minority use
ordinary softmax full attention (Multi-head Latent Attention, MLA). The two are
interleaved at a fixed **3:1 ratio** — three linear layers for every one full-
attention layer — which the paper finds recovers full-attention quality at a
fraction of the KV-cache and compute cost.

  • KDA layers carry positional information implicitly through their recurrence,
    so the full-attention layers need NO positional encoding. Hence the MLA layers
    here are NoPE (see multi_latent_attention/attention.py).
  • Every layer's channel mixer (FFN) is a DeepSeek-V3 / Moonlight-style MoE.

THIS FILE'S ONE DELIBERATE SUBSTITUTION
---------------------------------------
We replace KDA with **Gated DeltaNet-2** ("Decoupling Erase and Write in Linear
Attention", arXiv:2605.22791). Both are gated-delta-rule linear attentions with
fine-grained (channel-wise) gating; GDN-2's twist is a separate erase gate `b` and
write gate `w` instead of the single `beta` that KDA/GDN share. Everything else of
Kimi Linear — the 3:1 hybrid schedule, NoPE MLA, MoE FFN, pre-norm residual blocks
— is kept as in the paper. See gated_deltanet_2/layer.py for that token mixer.

BLOCK STRUCTURE (standard pre-norm transformer; Fig. 2)
-------------------------------------------------------
    x = x + TokenMixer(RMSNorm(x))     # TokenMixer = GDN-2 (linear) OR MLA (full)
    x = x + ChannelMixer(RMSNorm(x))   # ChannelMixer = MoE (or a dense SwiGLU MLP)

MODEL = Embed -> [DecoderLayer] * n_layers -> RMSNorm -> LM head.

TWO FORWARD MODES
-----------------
  • Training / full sequence:  model(input_ids)  — parallel, GDN-2 via its chunkwise
    core, MLA via a full causal-attention matrix.
  • Streaming / inference:     model.step(ids, caches) and model.generate(...)  —
    reuses per-layer state across calls so each new token is O(1) work for the GDN-2
    layers (fixed-size recurrent state) and O(context) for the few MLA layers (growing
    latent cache). See GatedDeltaNet2.step / GroupedQueryLatentAttention.step.
"""

# App. D.5: Xavier-uniform init with gain 2^{-2.5} (variance_scaling scale = gain² =
# 2^{-5}) for the embedding and LM head, replacing Flax NNX's defaults. The (small)
# embedding scale this produces is fine — RMSNorm renormalizes the residual stream.
_XAVIER = nnx.initializers.variance_scaling(2**-5, "fan_avg", "uniform")


# --------------------------------------------------------------------------- #
#  Configuration
#
#  Defaults are deliberately TINY so the whole model trains on a laptop CPU. The
#  paper's 48B-A3B numbers are quoted in comments for reference; only the *ratios*
#  and structure matter for understanding — scale up by raising the dims/layers.
# --------------------------------------------------------------------------- #
@dataclasses.dataclass
class KimiLinearConfig:
    vocab_size: int = 256  # paper: 160k; tiny here (byte-level demo)
    d_model: int = 256  # model width  (paper 1.3B: 2048)
    n_layers: int = 8  # depth        (paper 1.3B: 27)

    # --- Hybrid schedule: which layers are FULL attention (MLA) vs linear (GDN-2) ---
    # full_attn_period = 4 places one MLA layer every 4th layer (indices 3, 7, ...),
    # i.e. a 3:1 linear:full ratio — exactly Kimi Linear's hybrid recipe (Sec. 3.2).
    full_attn_period: int = 4

    # --- GDN-2 token mixer (the KDA replacement) — see gated_deltanet_2/layer.py ---
    gdn_num_heads: int = 4  # H key/query heads   (paper 1.3B: 16)
    gdn_head_k_dim: int = 64  # d_k                 (paper: 128)
    gdn_head_v_dim: int = 64  # d_v                 (paper: 128)
    gdn_num_v_heads: int | None = None  # H_v for GQA value heads; None -> = num_heads
    gdn_chunk_size: int = 64  # chunkwise block size C (paper App.: 64).
    #   NOTE: the GDN-2 chunkwise core requires every fed sequence length to be a
    #   multiple of this C (it reshapes L into L/C chunks). Keep seq_len % C == 0.
    gdn_conv_size: int = 4  # short-conv kernel width
    gdn_expanded_erase: bool = False  # erase gate in [0,2] (neg-eigenvalue variant)

    # --- MLA full-attention layers (NoPE) — see multi_latent_attention/attention.py ---
    mla_num_q_heads: int = 8  # query heads
    mla_num_kv_heads: int = 2  # KV/latent heads (GQA); q_heads must be a multiple
    mla_head_dim: int = 64  # per-head latent (rank) width
    max_seq_len: int = 512  # builds the causal mask; cap on trainable length

    # --- Channel mixer (FFN) ---
    moe_d_ff: int = 512  # per-expert hidden width (paper: 1408 at 1.3B)
    moe_n_routed: int = 8  # number of routed experts E (paper: 256)
    moe_n_shared: int = 1  # always-on shared experts
    moe_top_k: int = 2  # experts activated per token (paper: 8)
    mlp_d_ff: int = 768  # hidden width of the dense MLP fallback

    rms_eps: float = 1e-5


# --------------------------------------------------------------------------- #
#  One decoder block: pre-norm token mixer + pre-norm channel mixer, both residual.
#
#  The ONLY thing that varies across layers is the token mixer: GDN-2 (linear) on
#  most layers, MLA (full attention) on the 3:1 schedule. The channel mixer (MoE or
#  dense MLP) is the same kind on every layer — this matches Kimi Linear, where the
#  hybrid is in the *attention*, not the FFN.
# --------------------------------------------------------------------------- #
class DecoderLayer(nnx.Module):
    def __init__(self, cfg: KimiLinearConfig, layer_idx: int, *, rngs: nnx.Rngs):
        # 3:1 schedule: this layer is full-attention iff it is the last of its period.
        self.is_full_attn = (layer_idx + 1) % cfg.full_attn_period == 0

        # Pre-norm before the token mixer (Fig. 2). RMSNorm reused from the GDN-2 layer.
        self.norm1 = RMSNorm(cfg.d_model, eps=cfg.rms_eps, rngs=rngs)

        if self.is_full_attn:
            # Full attention: NoPE Multi-head Latent Attention (absorbed/GQA form).
            self.token_mixer = GroupedQueryLatentAttention(
                embed_dim=cfg.d_model,
                num_q_heads=cfg.mla_num_q_heads,
                num_kv_heads=cfg.mla_num_kv_heads,
                head_dim=cfg.mla_head_dim,
                dropout_rate=0.0,
                seq_length=cfg.max_seq_len,
                rngs=rngs,
            )
        else:
            # Linear attention: Gated DeltaNet-2 (the KDA substitute).
            self.token_mixer = GatedDeltaNet2(
                d_model=cfg.d_model,
                num_heads=cfg.gdn_num_heads,
                head_k_dim=cfg.gdn_head_k_dim,
                head_v_dim=cfg.gdn_head_v_dim,
                num_v_heads=cfg.gdn_num_v_heads,
                chunk_size=cfg.gdn_chunk_size,
                conv_size=cfg.gdn_conv_size,
                expanded_erase=cfg.gdn_expanded_erase,
                rngs=rngs,
            )

        # Pre-norm before the channel mixer.
        self.norm2 = RMSNorm(cfg.d_model, eps=cfg.rms_eps, rngs=rngs)

        # Channel mixer: MoE
        self.channel_mixer = GroupedGemmMoE(
            d_model=cfg.d_model,
            d_ff=cfg.moe_d_ff,
            n_routed=cfg.moe_n_routed,
            n_shared=cfg.moe_n_shared,
            top_k=cfg.moe_top_k,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> tuple[jax.Array, dict[str, jax.Array]]:
        """x: [B, L, d_model] -> (x, aux_or_None).

        `aux` carries the MoE load-balancing diagnostics the training loop
        needs (aux loss + per-expert token counts for the router-bias update).
        """
        # --- token mixing (residual, pre-norm) ---
        h = self.norm1(x)
        h = self.token_mixer(h)
        x = x + h

        # --- channel mixing (residual, pre-norm) ---
        y = self.norm2(x)
        m, aux = self.channel_mixer(y)
        x = x + m
        return x, aux

    def init_cache(self, batch_size: int, max_len: int, dtype=jnp.float32):
        """Per-layer streaming cache: a GDN2Cache (linear layer) or MLACache (MLA)."""
        return self.token_mixer.init_cache(batch_size, max_len, dtype)

    def step(self, x: jax.Array, cache):
        """Streaming forward for one block. x: [B, L, d_model] -> (x, new_cache).
        Only the token mixer is stateful; the channel mixer (MoE/MLP) is position-wise,
        so it needs no cache."""
        h = self.norm1(x)
        h, new_cache = self.token_mixer.step(
            h, cache
        )  # GDN-2 and MLA both expose .step
        x = x + h
        y = self.norm2(x)
        m, _aux = self.channel_mixer(y)
        x = x + m
        return x, new_cache


# --------------------------------------------------------------------------- #
#  The full model.
# --------------------------------------------------------------------------- #
class KimiLinear(nnx.Module):
    """Decoder-only Kimi Linear LM with a GDN-2 linear-attention backbone."""

    def __init__(self, cfg: KimiLinearConfig, *, rngs: nnx.Rngs):
        self.cfg = cfg
        # Token embedding table.
        self.embed = nnx.Embed(
            cfg.vocab_size, cfg.d_model, embedding_init=_XAVIER, rngs=rngs
        )
        # Stack of decoder blocks. NOTE: in Flax NNX a plain Python list of submodules
        # is not tracked as state — it must be wrapped in nnx.List(...).
        self.layers = nnx.List(
            [DecoderLayer(cfg, i, rngs=rngs) for i in range(cfg.n_layers)]
        )
        # Final pre-head norm + untied LM head (Moonlight/DeepSeek do not tie weights;
        # to tie, drop lm_head and use `x @ self.embed.embedding.value.T` instead).
        self.norm_f = RMSNorm(cfg.d_model, eps=cfg.rms_eps, rngs=rngs)
        self.lm_head = nnx.Linear(
            cfg.d_model, cfg.vocab_size, use_bias=False, kernel_init=_XAVIER, rngs=rngs
        )

    def __call__(
        self, input_ids: jax.Array, return_aux: bool = False
    ) -> tuple[jax.Array, dict[str, ArrayLike]]:
        """input_ids: int[B, L] -> logits[B, L, vocab]  (or (logits, aux) if return_aux).

        aux = {"aux_loss": scalar summed over MoE layers,
               "group_sizes": list of per-expert token counts, one entry per layer}.
        """
        x = self.embed(input_ids)  # [B, L, d_model]

        aux_loss: ArrayLike = 0.0
        group_sizes: list[
            ArrayLike
        ] = []  # one [E] vector per MoE layer, in layer order
        for layer in self.layers:
            x, aux = layer(x)

            aux_loss = aux_loss + aux["aux_loss"]
            group_sizes.append(aux["group_sizes"])

        x = self.norm_f(x)
        logits = self.lm_head(x)  # [B, L, vocab]

        return logits, {"aux_loss": aux_loss, "group_sizes": jnp.stack(group_sizes)}

    # ----------------------------------------------------------------------- #
    #  Streaming / inference.  Each layer carries its own cache (GDN-2: fixed-size
    #  recurrent state + conv state; MLA: growing latent cache).  Reusing them makes
    #  generation O(1) per token for the linear layers instead of re-reading history.
    # ----------------------------------------------------------------------- #
    def init_cache(
        self, batch_size: int, max_len: int | None = None, dtype=jnp.float32
    ) -> list:
        """Streaming caches for every layer. `max_len` (default cfg.max_seq_len) sizes
        the MLA latent buffers; GDN-2 layers ignore it (their state is fixed-size)."""
        max_len = max_len or self.cfg.max_seq_len
        return [layer.init_cache(batch_size, max_len, dtype) for layer in self.layers]

    def step(self, input_ids: jax.Array, caches: list) -> tuple[jax.Array, list]:
        """One streaming step. input_ids: int[B, L] (L = prompt length on prefill, or
        1 per decoded token). Returns (logits[B, L, vocab], new_caches)."""
        x = self.embed(input_ids)
        new_caches = []
        for layer, cache in zip(self.layers, caches):
            x, new_cache = layer.step(x, cache)
            new_caches.append(new_cache)
        x = self.norm_f(x)
        return self.lm_head(x), new_caches

    def generate(
        self, prompt_ids: jax.Array, max_new_tokens: int, max_len: int | None = None
    ) -> jax.Array:
        """Greedy autoregressive decode that REUSES each layer's state across steps.
        prompt_ids: int[B, P]. Returns the continuation int[B, max_new_tokens].

        Prefill consumes the whole prompt in one step (filling every layer's cache);
        each decode step then feeds back ONE token and carries the caches forward — the
        GDN-2 layers via their fixed-size recurrent state, the MLA layers via the
        growing latent cache. (Wrap `step` in nnx.jit for a fast decode loop.)"""
        B, P = prompt_ids.shape
        max_len = max_len or (P + max_new_tokens)
        caches = self.init_cache(B, max_len)
        logits, caches = self.step(prompt_ids, caches)  # prefill the prompt
        next_tok = jnp.argmax(logits[:, -1:], axis=-1)  # [B, 1] greedy
        outs = [next_tok]
        for _ in range(max_new_tokens - 1):
            logits, caches = self.step(next_tok, caches)  # decode one token
            next_tok = jnp.argmax(logits[:, -1:], axis=-1)
            outs.append(next_tok)
        return jnp.concatenate(outs, axis=1)  # [B, max_new_tokens]


def count_params(model: nnx.Module) -> int:
    """Total number of trainable parameters (sum of nnx.Param leaf sizes)."""
    return int(sum(x.size for x in jax.tree.leaves(nnx.state(model, nnx.Param))))


In [ ]:
"""
Configuration for the code-generation training/eval cycle.

A `TrainConfig` bundles together:
  * `model`   - the `KimiLinearConfig` for the network (see kimi_linear_gdn2.py),
  * tokenizer / data / optimization / evaluation hyper-parameters.

Two presets are provided:

  * "tiny"  - deliberately small; the WHOLE cycle (tokenizer -> train -> pass@k)
              runs on a laptop CPU in a few minutes. Use it to verify the plumbing.
  * "small" - a ~200M-parameter (total) MoE model sized for a single GPU. Real (if
              modest) code-generation training; expect to actually need an accelerator.

Everything is plain dataclasses so a run is fully described (and serializable) by
one `TrainConfig`. CLI flags in train.py/evaluate.py override individual fields.
"""

from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field

# Special tokens shared by the tokenizer, data pipeline and sampler.
PAD_TOKEN = "<|pad|>"
EOS_TOKEN = "<|endoftext|>"
SPECIAL_TOKENS = [PAD_TOKEN, EOS_TOKEN]


@dataclass
class TrainConfig:
    name: str = "small"

    # --- model ---------------------------------------------------------------
    # vocab_size is overwritten at load time to match the trained tokenizer.
    model: KimiLinearConfig = field(default_factory=KimiLinearConfig)

    # --- tokenizer -----------------------------------------------------------
    vocab_size: int = 16000          # BPE target vocab (incl. special tokens)
    tokenizer_path: str = "runs/small/tokenizer.json"

    # --- data ----------------------------------------------------------------
    dataset: str = "mbpp"            # only "mbpp" is wired up here
    mbpp_config: str = "full"        # "full" (374 train) or "sanitized" (120 train)
    train_seq_len: int = 256         # padded length per example; MUST be a multiple
    #                                  of model.gdn_chunk_size (chunkwise core needs it)

    # --- optional Phase-1 pretraining (plain LM on a code corpus, before MBPP SFT) -
    # `pretrain_corpus` is either a LOCAL DIRECTORY of source files, or a HuggingFace
    # dataset id streamed via the `pretrain_hf_*` knobs. None -> SFT only.
    pretrain_corpus: str | None = None
    pretrain_hf_field: str = "content"     # text column for a HF dataset
    pretrain_hf_name: str | None = None    # HF config/name (e.g. "default")
    pretrain_hf_data_dir: str | None = None  # HF data_dir (e.g. "data/python")
    pretrain_max_docs: int = 5000          # cap #documents read (keeps it in RAM)
    pretrain_epochs: int = 1
    pretrain_max_steps: int | None = None  # overrides pretrain_epochs if set
    pretrain_lr: float | None = None       # defaults to `lr` if None

    # --- optimization --------------------------------------------------------
    batch_size: int = 16             # micro-batch size (sequences per forward)
    grad_accum: int = 2              # effective batch = batch_size * grad_accum
    epochs: int = 30                 # passes over the (small) MBPP train split
    max_steps: int | None = None     # if set, overrides epochs
    lr: float = 3e-4
    min_lr_ratio: float = 0.1        # final LR = lr * min_lr_ratio (cosine floor)
    warmup_ratio: float = 0.03       # warmup steps = warmup_ratio * total_steps
    weight_decay: float = 0.1        # applied to >=2-D kernels only (not norms/biases)
    grad_clip: float = 1.0
    adam_b1: float = 0.9
    adam_b2: float = 0.95
    router_bias_lr: float = 1e-3     # aux-loss-free MoE balancing step (see moe.py)

    # --- precision -----------------------------------------------------------
    # The model forces fp32 in its numerically sensitive paths (decay, RMSNorm,
    # MoE aux). "bfloat16" here sets the matmul accumulation precision lever, a
    # safe single-GPU speed-up; master weights stay fp32. See README "Precision".
    matmul_precision: str = "highest"   # "highest" (fp32) | "high" | "bfloat16"

    # --- logging / checkpointing --------------------------------------------
    out_dir: str = "runs/small"
    log_every: int = 10
    eval_every: int = 200            # steps between perplexity evals
    passk_every: int = 0             # steps between (slow) pass@k evals; 0 = only at end
    ckpt_every: int = 200
    keep_ckpts: int = 3
    seed: int = 0

    # --- evaluation ----------------------------------------------------------
    eval_max_problems: int = 0       # 0 = all MBPP test problems
    eval_n_samples: int = 5          # samples per problem (for pass@k)
    eval_ks: tuple[int, ...] = (1, 5)
    eval_temperature: float = 0.2
    eval_top_p: float = 0.95
    eval_max_new_tokens: int = 256
    eval_timeout: float = 8.0        # seconds per unit-test execution

    def resolved_vocab(self) -> int:
        return self.model.vocab_size


# --------------------------------------------------------------------------- #
#  Presets
# --------------------------------------------------------------------------- #
def _tiny() -> TrainConfig:
    """CPU smoke-test scale: the full cycle runs in minutes, proves the plumbing."""
    model = KimiLinearConfig(
        vocab_size=2048,
        d_model=128,
        n_layers=4,
        full_attn_period=4,          # one MLA layer (index 3); rest GDN-2
        gdn_num_heads=2,
        gdn_head_k_dim=32,
        gdn_head_v_dim=32,
        gdn_chunk_size=32,
        mla_num_q_heads=4,
        mla_num_kv_heads=2,
        mla_head_dim=32,
        max_seq_len=320,
        moe_d_ff=128,
        moe_n_routed=4,
        moe_n_shared=1,
        moe_top_k=2,
        mlp_d_ff=256,
    )
    return TrainConfig(
        name="tiny",
        model=model,
        vocab_size=2048,
        tokenizer_path="runs/tiny/tokenizer.json",
        mbpp_config="sanitized",
        train_seq_len=288,           # 288 % 32 == 0; long enough for full solutions
        batch_size=8,
        grad_accum=1,
        epochs=8,
        lr=5e-4,
        warmup_ratio=0.05,
        out_dir="runs/tiny",
        log_every=2,
        eval_every=20,
        ckpt_every=40,
        eval_max_problems=10,
        eval_n_samples=3,
        eval_ks=(1, 3),
        eval_max_new_tokens=128,
    )


def _small() -> TrainConfig:
    """~200M-param (total) MoE model for a single GPU; far fewer active per token
    (top-2 of 8 experts), so compute/step is well under the parameter count."""
    model = KimiLinearConfig(
        vocab_size=16000,
        d_model=512,
        n_layers=12,
        full_attn_period=4,          # 3 GDN-2 : 1 MLA
        gdn_num_heads=8,
        gdn_head_k_dim=64,
        gdn_head_v_dim=64,
        gdn_chunk_size=64,
        mla_num_q_heads=8,
        mla_num_kv_heads=2,
        mla_head_dim=64,
        max_seq_len=512,
        moe_d_ff=1024,
        moe_n_routed=8,
        moe_n_shared=1,
        moe_top_k=2,
        mlp_d_ff=2048,
    )
    return TrainConfig(
        name="small",
        model=model,
        vocab_size=16000,
        tokenizer_path="runs/small/tokenizer.json",
        mbpp_config="full",
        train_seq_len=256,           # 256 % 64 == 0
        batch_size=16,
        grad_accum=2,
        epochs=40,
        lr=3e-4,
        out_dir="runs/small",
        matmul_precision="bfloat16",
        eval_every=200,
        ckpt_every=200,
        eval_n_samples=5,
        eval_ks=(1, 5),
    )


PRESETS = {"tiny": _tiny, "small": _small}


def get_preset(name: str) -> TrainConfig:
    if name not in PRESETS:
        raise ValueError(f"unknown preset {name!r}; choose from {list(PRESETS)}")
    return PRESETS[name]()


def as_dict(cfg: TrainConfig) -> dict:
    """Flat-ish dict for JSON logging (model config nested under 'model')."""
    d = dataclasses.asdict(cfg)
    return d


In [ ]:
"""
Checkpointing for the Flax NNX model, via Orbax.

We persist only the model *state* (the pytree of parameter + variable arrays from
`nnx.split`). To restore we rebuild the module structure abstractly with
`nnx.eval_shape` (no host/device allocation), restore the arrays into that
template, and `nnx.merge` graph-def + state back into a live module. This is the
pattern recommended in the Flax NNX docs, and it means a checkpoint is portable as
long as the `KimiLinearConfig` used to rebuild matches.

A tiny JSON sidecar records step / metric so we can pick the best checkpoint.
"""

from __future__ import annotations

import json
import os
import shutil
import warnings
from typing import Any

import flax.nnx as nnx
import orbax.checkpoint as ocp

# Single-device restore: Orbax warns that no sharding was provided. Benign here.
warnings.filterwarnings("ignore", message=".*Sharding info not provided.*")

from kimi_linear_gdn2 import KimiLinear, KimiLinearConfig


def _abs(path: str) -> str:
    return os.path.abspath(path)


def save_checkpoint(model: KimiLinear, path: str, meta: dict[str, Any] | None = None) -> None:
    """Save model state (overwriting any existing checkpoint at `path`)."""
    path = _abs(path)
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)

    _, state = nnx.split(model)
    ckptr = ocp.StandardCheckpointer()
    ckptr.save(os.path.join(path, "state"), state)
    ckptr.wait_until_finished()

    with open(os.path.join(path, "meta.json"), "w") as f:
        json.dump(meta or {}, f, indent=2, default=str)


def load_checkpoint(cfg: KimiLinearConfig, path: str) -> KimiLinear:
    """Rebuild a KimiLinear from a checkpoint produced by `save_checkpoint`."""
    path = _abs(path)
    state_dir = os.path.join(path, "state")
    if not os.path.exists(state_dir):
        raise FileNotFoundError(f"no checkpoint state at {state_dir!r}")

    abstract = nnx.eval_shape(lambda: KimiLinear(cfg, rngs=nnx.Rngs(0)))
    graphdef, abstract_state = nnx.split(abstract)

    ckptr = ocp.StandardCheckpointer()
    restored = ckptr.restore(state_dir, abstract_state)
    return nnx.merge(graphdef, restored)


def read_meta(path: str) -> dict[str, Any]:
    meta_path = os.path.join(_abs(path), "meta.json")
    if not os.path.exists(meta_path):
        return {}
    with open(meta_path) as f:
        return json.load(f)
